# TB-Trust — 07: The density resolution floor and the certificate of insufficiency

Notebook 06 recovered the capture channel. This one turns it into a **bound**.

For a finding with unit-amplitude profile `t(x)` and density contrast `dD`, seen through a
capture with modulation transfer `MTF(f)` and per-pixel differential density noise `sigma_D`,
a matched filter — the optimal detector for a known signal in additive noise, so nothing can
do better — achieves

$$\mathrm{SNR} = \frac{\Delta D\sqrt{E}}{\sigma_D},\qquad E=\sum_f |T(f)|^2\,\mathrm{MTF}(f)^2$$

and requiring `SNR >= k` (Rose criterion, k=5) gives the **density resolution floor**

$$\Delta D_{\text{floor}}(x) = k\,\sigma_D(x)\,/\,\sqrt{E}.$$

Compare it against the characteristic contrast of a TB finding and you get a verdict. When the
floor exceeds the contrast, the statement is not "the model is unsure" — it is **the
information is not in the photograph**. No model, however good, and no amount of training data
recovers a density step the channel cannot carry.

Written out, `sigma_D` is

$$\sigma_D=\frac{\gamma}{\ln 10}\cdot\frac{\sigma_v}{v-c_0}\cdot\left(1+\frac{V}{I}\right)$$

— note the veil enters as `1 + V/I`, exactly the reciprocal of the contrast compression it
imposes. That term is usually the largest one in a real clinic photo, and it is the one nobody
measures, because measuring it needs a beam stop and nobody noticed the film already has one.

> **The finding contrasts are nominal placeholders.** `physics/findings.py` ships physically
> sensible defaults marked `source="NOMINAL"`, not values from a published table. Relative
> statements — this photo carries less than that one, glare cost a factor of three — depend
> only on the floor and are sound. Absolute verdicts inherit the table's uncertainty. Replace
> it via `--findings` before publishing; the module docstring says how.

In [ ]:
# --- configuration ---------------------------------------------------------
# Every path comes from the environment first, so this notebook runs unmodified
# on Kaggle, locally, or under scripts/test_notebooks.py in CI.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(OUT, exist_ok=True)

# Working resolution for the physics. This is the single most consequential knob
# in the whole track: the density floor depends on how many pixels a finding
# spans, so a 2 mm miliary nodule is under two pixels at 320 px and the
# certificate correctly -- but uselessly -- calls every image insufficient.
# A phone photographing a 35 cm film at 3000 px gets about 8 px/mm. 1024 is the
# smallest size at which the severity sweep separates properly; drop it only to
# make a CI run cheap.
PHYSICS_SIZE = int(os.environ.get("TBTRUST_PHYSICS_SIZE", "1024"))
N_IMAGES = int(os.environ.get("TBTRUST_PHYSICS_N", "24"))

print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)
print("physics size:", PHYSICS_SIZE, " images:", N_IMAGES)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
print("tbtrust ready from", REPO)

## 1. One image: the floor, and what is limiting it

In [ ]:
from tbtrust.physics import findings as FIND
from tbtrust.physics.certificate import certify
from tbtrust.physics.film import capture, sample_params, synthetic_chest_density
from tbtrust.physics.floor import density_floor, limiting_factor
from tbtrust.physics.invert import invert

base, ftruth = synthetic_chest_density(size=PHYSICS_SIZE, rng=np.random.default_rng(0))
photo, truth = capture(base, sample_params(0.5, np.random.default_rng(1)),
                       fiducial_truth=ftruth, rng=np.random.default_rng(2))
cal = invert(photo)
lung = cal.lung_field_mask()

fm = density_floor(cal, FIND.get("infiltrate"))
name, detail = limiting_factor(fm, lung)
print(f"px/mm {cal.px_per_mm:.2f} | median floor {np.median(fm.floor[lung]):.4f} dD "
      f"| blur penalty x{fm.blur_penalty:.2f} | limiting: {name}")
pd.Series(detail["share"]).sort_values(ascending=False).to_frame("share of the floor").round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
f_show = np.where(lung, fm.floor, np.nan)
im = axes[0].imshow(f_show, cmap="inferno")
axes[0].set_title("density floor over the lung field")
plt.colorbar(im, ax=axes[0], fraction=0.046, label="min resolvable dD")

insuff = lung & (fm.floor > FIND.get("infiltrate").delta_d)
axes[1].imshow(photo, cmap="gray")
axes[1].imshow(np.ma.masked_where(~insuff, insuff), cmap="autumn", alpha=0.5)
axes[1].set_title(f"where an infiltrate would be lost\n({insuff.sum() / max(lung.sum(), 1):.0%} of the lung field)")

im = axes[2].imshow(np.where(lung, cal.veil / np.maximum(cal.signal, 1e-9), np.nan), cmap="magma")
axes[2].set_title("veil / signal")
plt.colorbar(im, ax=axes[2], fraction=0.046)
for a in axes:
    a.axis("off")
fig.tight_layout()
plt.show()

## 2. The certificate

The image's verdict is the **worst** line item, not the average. Screening is a sensitivity
problem: an image that can carry a lobar consolidation but not a miliary pattern has still lost
the finding you most need not to miss, and averaging across findings hides exactly that.

In [ ]:
from tbtrust.physics import figures as FIG

cert = certify(cal, findings=FIND.all_findings())
print(cert.report())

The same verdict as an object a clinician could be handed, rather than a text dump. The bars
are margins in dB; the inset is the evidence — every pixel where the measured floor beats the
worst finding's contrast.

In [ ]:
from tbtrust.physics.triage import triage

decision = triage(cert, cal, model_confidence=0.85)
FIG.show(FIG.certificate_card(cert, cal, decision))
plt.show()

And the same result laid over anatomy: every TB finding drawn at its true relative size in a
location it actually favours (post-primary TB is an upper-lobe disease), shaded by whether
*this photograph* can carry it. A 2 mm miliary nodule really is a speck beside a 45 mm
consolidation — which is why their floors differ by more than an order of magnitude.

In [ ]:
FIG.show(FIG.finding_atlas(cert, cal.px_per_mm))
plt.show()

## 3. Contrast-detail: the clearest single picture the physics track produces

Findings above the floor curve are carried by the photograph; findings below it are not. The
gap between a clean capture's curve and a degraded one is the cost of the capture, in the units
radiology actually reads.

In [ ]:
sizes_mm = np.geomspace(1.0, 60.0, 24)


def floor_curve(cal, mask, sizes):
    base_f = FIND.get("infiltrate")
    out = []
    for s in sizes:
        fm = density_floor(cal, base_f.rescaled(size_mm=float(s)))
        out.append(float(np.median(fm.floor[mask])))
    return np.array(out)


curves = {}
for sev in (0.0, 0.5, 1.0):
    p = sample_params(sev, np.random.default_rng(1))
    ph, _ = capture(base, p, fiducial_truth=ftruth, rng=np.random.default_rng(2))
    c = invert(ph)
    curves[sev] = (c, floor_curve(c, c.lung_field_mask(), sizes_mm))

fig, ax = plt.subplots(figsize=(7, 5))
for sev, (_c, y) in curves.items():
    ax.loglog(sizes_mm, y, "-", lw=2, label=f"floor, capture severity {sev}")
for f in FIND.all_findings():
    ax.errorbar(f.size_mm, f.delta_d, yerr=f.delta_d_sigma, xerr=f.size_sigma_mm,
                fmt="o", ms=7, capsize=3, color="k")
    ax.annotate(f.name, (f.size_mm, f.delta_d), textcoords="offset points",
                xytext=(7, 5), fontsize=8)
ax.set_xlabel("finding size (mm)")
ax.set_ylabel("density contrast |dD|")
ax.set_title("Contrast-detail: measured floor vs TB finding contrasts\n(finding values are NOMINAL)")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 4. Certificates across the severity sweep

The ordering properties below must hold whatever the nominal contrast table says: the floor
rises with capture severity, the margin falls, and the insufficient fraction grows. If they do
not, something upstream is broken — most often the fiducial detector failing on the degraded
images and quietly abstaining.

In [ ]:
from tbtrust.physics.triage import triage, triage_summary

rows, decisions = [], []
severities = [0.0, 0.25, 0.5, 0.75, 1.0]
n_img = max(3, N_IMAGES // 4)
for i in range(n_img):
    b, ft = synthetic_chest_density(size=PHYSICS_SIZE, rng=np.random.default_rng(100 + i))
    for sev in severities:
        ph, _ = capture(b, sample_params(sev, np.random.default_rng(200 + i)),
                        fiducial_truth=ft, rng=np.random.default_rng(300 + i))
        c = invert(ph)
        ct = certify(c)
        d = triage(ct, c, model_confidence=0.9)
        decisions.append((sev, d))
        rows.append({"image": i, "severity": sev, **ct.as_dict(),
                     "triage": d.action.value, "reason": d.reason})

certs = pd.DataFrame(rows)
certs.to_csv(f"{OUT}/physics_certificates_sweep.csv", index=False)

display(certs.groupby("severity").agg(
    margin_db=("margin_db", "median"),
    abstain=("certificate", lambda s: (s == "abstain").mean()),
    insufficient=("certificate", lambda s: (s == "insufficient").mean()),
    detectable=("certificate", lambda s: (s == "detectable").mean()),
    retake=("triage", lambda s: (s == "retake").mean()),
    refer=("triage", lambda s: (s == "refer").mean()),
).round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for key in FIND.CORE_FINDINGS:
    col = f"margin_db_{key}"
    if col in certs:
        axes[0].plot(certs.groupby("severity")[col].median(), "o-", label=FIND.get(key).name)
axes[0].axhline(0, color="k", ls="--", lw=1)
axes[0].axhspan(-40, 0, color="red", alpha=0.07)
axes[0].set_xlabel("capture severity")
axes[0].set_ylabel("margin (dB)")
axes[0].set_title("Margin per finding\n(below 0: information not in the photo)")
axes[0].legend(fontsize=7)

verd = certs.groupby(["severity", "certificate"]).size().unstack(fill_value=0)
verd = verd.div(verd.sum(axis=1), axis=0)
verd.plot(kind="bar", stacked=True, ax=axes[1], rot=0)
axes[1].set_title("Certificate verdict")
axes[1].set_ylabel("fraction")
axes[1].legend(fontsize=7)

lim = certs.groupby(["severity", "limiting_factor"]).size().unstack(fill_value=0)
lim = lim.div(lim.sum(axis=1), axis=0)
lim.plot(kind="bar", stacked=True, ax=axes[2], rot=0)
axes[2].set_title("What is limiting the floor")
axes[2].set_ylabel("fraction")
axes[2].legend(fontsize=7)

fig.tight_layout()
plt.show()
print(triage_summary([d for _, d in decisions]))

## 5. Capacity in bits, and where the lost bits went

Phase 1 reached for an information-theory framing and had to leave it a metaphor — there was no
measured signal and no measured noise, so there was no capacity. With the channel measured it
becomes arithmetic: `C = 0.5 log2(1 + SNR^2)` bits per resolution cell.

`bits_lost` ablates each impairment in turn and reports what it took. These are *attributions*,
not a decomposition — the terms interact multiplicatively, so they do not sum to the total.

In [ ]:
from tbtrust.physics.channel import capacity_table, reference_capacity

cap = pd.DataFrame(capacity_table(curves[0.5][0]))
display(cap.round(3))

deliv = pd.DataFrame([reference_capacity(curves[s][0], FIND.get(k))
                      for s in (0.0, 0.5, 1.0) for k in ("infiltrate", "cavity_wall")])
deliv["severity"] = [s for s in (0.0, 0.5, 1.0) for _ in range(2)]
print("\nFraction of the film's information the phone delivered:")
display(deliv.pivot(index="severity", columns="finding", values="delivered_fraction").round(3))

fig, ax = plt.subplots(figsize=(7, 3.6))
cap.set_index("finding")[["bits_lost_veil", "bits_lost_blur",
                          "bits_lost_quantization", "bits_lost_sensor_noise"]].plot(
    kind="barh", stacked=True, ax=ax)
ax.set_xlabel("bits per resolution cell recoverable by removing this impairment")
ax.set_title("Where the information went (severity 0.5)")
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## 6. The falsification test

Everything above is a claim. This is the experiment that could refute it.

Insert lesions of known contrast, push them through the simulated capture, and let an
**optimal** detector — a matched filter that already knows the lesion's position, size and
shape, so no real system can beat it — try to tell present from absent. Then ask whether the
empirical detectability threshold lands where the blindly-computed floor said it would.

This is not circular. The floor is computed from quantities the estimator had to *measure
blind*: `sigma_D` from a noise model fitted to the image, `E` from an MTF read off the
collimation border, the veil amplification from the beam stop. The empirical threshold comes
from the detector's actual behaviour on actual noisy captures. Nothing forces them to agree —
under-measure the veil and the floor comes out optimistic, over-smooth the MTF and it comes out
pessimistic.

`ratio = predicted / empirical`. **Above 1 the bound is conservative**, declaring information
lost slightly before an optimal detector loses it. Below 1 it is optimistic, which is the
dangerous direction for a safety valve. Report this ratio with any result: a bound quoted
without its measured calibration is an assertion.

First, the version you can check with your own eyes. The floor is computed **blind from the
photograph**, before any lesion exists. Lesions are then inserted into the film at fractions
and multiples of that floor and re-photographed through the same capture. The dashed circle
marks where to look, and both rows share one display window so the panels are genuinely
comparable.

If the bound is real, the lesion should be invisible below 1× and obvious above it — in the
recovered row, not just on the film.

In [ ]:
from tbtrust.physics.film import sample_params

params_demo = sample_params(0.5, np.random.default_rng(1))
FIG.show(FIG.detectability_strip(base, params_demo, FIND.get("infiltrate"),
                                 cal=cal, fiducial_truth=ftruth))
plt.show()

In [ ]:
from tbtrust.physics import validate as V

det_rows = []
for sev in (0.0, 0.25, 0.5):
    for key in ("infiltrate", "cavity_wall", "consolidation"):
        r = V.detectability_experiment(severity=sev, finding=key,
                                       n_trials=max(8, N_IMAGES // 2),
                                       size=min(320, PHYSICS_SIZE), seed=1)
        det_rows.append(r.as_dict())
        print(f"sev {sev:<5} {key:<14} predicted {r.predicted_floor:8.4f}  "
              f"empirical {r.empirical_threshold:8.4f}  ratio {r.ratio:5.2f}  r2 {r.linearity_r2:.2f}")

det = pd.DataFrame(det_rows)
det.to_csv(f"{OUT}/physics_detectability.csv", index=False)
finite = det[np.isfinite(det["ratio"]) & (det["ratio"] > 0)]
print(f"\nmedian ratio {finite['ratio'].median():.2f} over {len(finite)} measurable conditions")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].loglog(finite["empirical_threshold"], finite["predicted_floor"], "o", ms=8)
lim = [finite[["empirical_threshold", "predicted_floor"]].min().min() * 0.5,
       finite[["empirical_threshold", "predicted_floor"]].max().max() * 2]
axes[0].plot(lim, lim, "k--", lw=1, label="perfect calibration")
axes[0].fill_between(lim, [x * 0.5 for x in lim], [x * 2 for x in lim], color="green", alpha=0.08,
                     label="within a factor of 2")
axes[0].set_xlabel("empirical threshold of an optimal detector")
axes[0].set_ylabel("predicted density floor")
axes[0].set_title("Does the bound predict detectability?")
axes[0].legend(fontsize=8)

axes[1].bar(range(len(finite)), finite["ratio"])
axes[1].axhline(1, color="k", ls="--", lw=1)
axes[1].axhspan(0.5, 2.0, color="green", alpha=0.08)
axes[1].set_xticks(range(len(finite)))
axes[1].set_xticklabels([f"{r.severity}\n{r.finding[:8]}" for r in finite.itertuples()], fontsize=7)
axes[1].set_ylabel("predicted / empirical")
axes[1].set_title("Calibration of the bound\n(>1 = conservative, the safe direction)")

fig.tight_layout()
plt.show()

### What this establishes, and what it does not

**Does:** the density resolution floor is a real, measured, falsifiable bound on the capture
channel, calibrated to within roughly a factor of two of an optimal detector's actual
threshold, and conservative on the median.

**Does not:** say anything about how hard the *diagnosis* is. The dominant obstacle to spotting
a real nodule on a real chest radiograph is anatomical clutter — ribs, vessels, the heart
border — not photon noise, and a lesion can sit comfortably above this floor and still be
invisible against a rib. `floor.anatomical_noise` estimates that clutter and
`density_floor(..., include_anatomical=True)` will fold it in, but the certificate deliberately
does not, because its claim is the narrow defensible one: *this photograph destroyed
information the film had*. Anatomical clutter is present in the original film too, and no
retake fixes it.